# Product Label Prediction
## Classical vectorization comparison + DistilBERT multi-label benchmark

This notebook:

1. Loads pair-level labels from `pair_level_gpt41mini_clean.csv`
2. Loads raw Reddit text from `reddit_comments.csv` and `reddit_posts.csv`
3. Merges `ext_id` back to original text
4. Builds a **text-level 5-label** dataset:
   - flower
   - oil
   - gummies
   - vape
   - topical
5. Compares several **classical vectorization/model pipelines**
6. Tunes thresholds on the validation set
7. Adds a **DistilBERT multi-label classifier** as the final benchmark
8. Reports the best classical model and the best overall model

This notebook is written to be robust to small schema/path differences.

In [ ]:
# =========================
# 1. Imports
# =========================
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# =========================
# 2. Paths
# =========================
PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis")
FINAL_PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/FInal Project")

PAIR_PATH = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/data_processing/data/final/pair_level_gpt41mini_clean.csv")
COMMENTS_PATH = FINAL_PROJECT_ROOT / "reddit_comments.csv"
POSTS_PATH = FINAL_PROJECT_ROOT / "reddit_posts.csv"

OUTPUT_DIR = PROJECT_ROOT / "Modeling" / "product_label_prediction_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PAIR_PATH exists:", PAIR_PATH.exists(), PAIR_PATH)
print("COMMENTS_PATH exists:", COMMENTS_PATH.exists(), COMMENTS_PATH)
print("POSTS_PATH exists:", POSTS_PATH.exists(), POSTS_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# =========================
# 3. Helper functions
# =========================
LABELS = ["flower", "oil", "gummies", "vape", "topical"]

def standardize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_product(x):
    x = standardize_text(x).lower()
    mapping = {
        "gummy": "gummies",
        "gummies": "gummies",
        "flower": "flower",
        "oil": "oil",
        "vape": "vape",
        "topical": "topical",
    }
    return mapping.get(x, x)

def combine_post_text(title, selftext):
    title = standardize_text(title)
    selftext = standardize_text(selftext)
    if title and selftext:
        return f"{title} {selftext}"
    return title or selftext

def evaluate_multilabel(y_true, y_pred, label_names, model_name):
    summary = {
        "model": model_name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred)
    }
    per_label = pd.DataFrame({
        "label": label_names,
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=None, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=None, zero_division=0)
    })
    return summary, per_label

def tune_thresholds(y_true, y_score, label_names, grid=None):
    if grid is None:
        grid = np.arange(0.1, 0.91, 0.05)
    best_thresholds = {}
    for j, label in enumerate(label_names):
        best_t = 0.5
        best_f1 = -1
        yt = y_true[:, j]
        ys = y_score[:, j]
        for t in grid:
            yp = (ys >= t).astype(int)
            score = f1_score(yt, yp, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = float(t)
        best_thresholds[label] = best_t
    return best_thresholds

def apply_thresholds(y_score, thresholds, label_names):
    out = np.zeros_like(y_score, dtype=int)
    for j, label in enumerate(label_names):
        out[:, j] = (y_score[:, j] >= thresholds[label]).astype(int)
    return out

def linear_svc_scores(ovr_model, X):
    scores = []
    for est in ovr_model.estimators_:
        s = est.decision_function(X)
        s = 1 / (1 + np.exp(-s))
        scores.append(s)
    return np.column_stack(scores)

def find_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

In [ ]:
# =========================
# 4. Load pair labels and raw Reddit files
# =========================
pair_df = pd.read_csv(PAIR_PATH)
comments_df = pd.read_csv(COMMENTS_PATH)
posts_df = pd.read_csv(POSTS_PATH)

print("pair_df:", pair_df.shape)
print("comments_df:", comments_df.shape)
print("posts_df:", posts_df.shape)

display(pair_df.head())
display(comments_df.head(2))
display(posts_df.head(2))

In [ ]:
# =========================
# 5. Build text lookup from comments and posts
# =========================
comments_use = comments_df.copy()
comments_use["ext_id"] = comments_use["id"].astype(str)
comments_use["text"] = comments_use["body"].fillna("").map(standardize_text)
comments_use["source"] = "comment"
comment_keep = ["ext_id", "text", "source"]
for c in ["subreddit", "created_utc", "searched_keyword", "score", "parent_id", "link_id"]:
    if c in comments_use.columns:
        comment_keep.append(c)
comments_use = comments_use[comment_keep].drop_duplicates(subset=["ext_id"])

posts_use = posts_df.copy()
posts_use["ext_id"] = posts_use["id"].astype(str)
posts_use["text"] = [
    combine_post_text(t, s) for t, s in zip(posts_use.get("title", ""), posts_use.get("selftext", ""))
]
posts_use["source"] = "post"
post_keep = ["ext_id", "text", "source"]
for c in ["subreddit", "created_utc", "searched_keyword", "score", "num_comments"]:
    if c in posts_use.columns:
        post_keep.append(c)
posts_use = posts_use[post_keep].drop_duplicates(subset=["ext_id"])

text_lookup = pd.concat([comments_use, posts_use], ignore_index=True).drop_duplicates(subset=["ext_id"], keep="first")
text_lookup["ext_id"] = text_lookup["ext_id"].astype(str)
text_lookup["text"] = text_lookup["text"].fillna("").map(standardize_text)

print(text_lookup.shape)
display(text_lookup.head())

In [ ]:
# =========================
# 6. Merge pair labels back to text
# =========================
pair_df["ext_id"] = pair_df["ext_id"].astype(str)
pair_df["product"] = pair_df["product"].map(normalize_product)

pair_df = pair_df[pair_df["product"].isin(LABELS)].copy()

merged = pair_df.merge(text_lookup, on="ext_id", how="left")

print("Merged shape:", merged.shape)
print("Missing text rows:", merged["text"].isna().sum())
display(merged.head())

In [ ]:
# =========================
# 7. Build text-level 5-label dataset
# =========================
# Aggregate products per ext_id/text
agg = (
    merged.groupby("ext_id")
    .agg({
        "product": lambda x: sorted(set([p for p in x if p in LABELS])),
        "text": "first",
        "source": "first",
        **({c: "first" for c in ["subreddit", "created_utc", "searched_keyword"] if c in merged.columns})
    })
    .reset_index()
)

agg["text"] = agg["text"].fillna("").map(standardize_text)
agg = agg[agg["text"].str.len() > 0].copy()

for label in LABELS:
    agg[label] = agg["product"].apply(lambda xs: int(label in xs))

agg["n_labels"] = agg[LABELS].sum(axis=1)

text_df = agg[["ext_id", "text", "source"] + [c for c in ["subreddit", "created_utc", "searched_keyword"] if c in agg.columns] + LABELS + ["n_labels"]].copy()

print(text_df.shape)
display(text_df.head())
print(text_df[LABELS].sum().sort_values(ascending=False))
print("Rows with multiple labels:", (text_df["n_labels"] > 1).sum())

In [ ]:
# =========================
# 8. Train/val/test split by ext_id
# =========================
train_df, temp_df = train_test_split(text_df, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

X_train = train_df["text"].tolist()
X_val   = val_df["text"].tolist()
X_test  = test_df["text"].tolist()

y_train = train_df[LABELS].values.astype(int)
y_val   = val_df[LABELS].values.astype(int)
y_test  = test_df[LABELS].values.astype(int)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Train label counts:")
display(train_df[LABELS].sum())

## Classical models

Below we compare several **non-transformer** vectorization/model setups.  
Each one is threshold-tuned on the validation set before final evaluation on the test set.

In [ ]:
# =========================
# 9. Classical Model A: CountVectorizer + LinearSVC
# =========================
count_svc = Pipeline([
    ("vec", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])

count_svc.fit(X_train, y_train)

Xv = count_svc.named_steps["vec"].transform(X_val)
Xt = count_svc.named_steps["vec"].transform(X_test)

val_scores = linear_svc_scores(count_svc.named_steps["clf"], Xv)
test_scores = linear_svc_scores(count_svc.named_steps["clf"], Xt)

count_thresholds = tune_thresholds(y_val, val_scores, LABELS)
count_pred = apply_thresholds(test_scores, count_thresholds, LABELS)

count_summary, count_per_label = evaluate_multilabel(y_test, count_pred, LABELS, "Count + LinearSVC")
display(pd.DataFrame([count_summary]))
display(count_per_label)
print("Thresholds:", count_thresholds)

In [ ]:
# =========================
# 10. Classical Model B: HashingVectorizer + LinearSVC
# =========================
hash_svc = Pipeline([
    ("vec", HashingVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        alternate_sign=False,
        n_features=2**18
    )),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])

hash_svc.fit(X_train, y_train)

Xv = hash_svc.named_steps["vec"].transform(X_val)
Xt = hash_svc.named_steps["vec"].transform(X_test)

val_scores = linear_svc_scores(hash_svc.named_steps["clf"], Xv)
test_scores = linear_svc_scores(hash_svc.named_steps["clf"], Xt)

hash_thresholds = tune_thresholds(y_val, val_scores, LABELS)
hash_pred = apply_thresholds(test_scores, hash_thresholds, LABELS)

hash_summary, hash_per_label = evaluate_multilabel(y_test, hash_pred, LABELS, "Hashing + LinearSVC")
display(pd.DataFrame([hash_summary]))
display(hash_per_label)
print("Thresholds:", hash_thresholds)

In [ ]:
# =========================
# 11. Classical Model C: Character n-grams + Logistic Regression
# =========================
char_logreg = Pipeline([
    ("vec", CountVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        lowercase=True
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear"
        )
    ))
])

char_logreg.fit(X_train, y_train)

Xv = char_logreg.named_steps["vec"].transform(X_val)
Xt = char_logreg.named_steps["vec"].transform(X_test)

val_scores = np.column_stack([
    est.predict_proba(Xv)[:, 1] for est in char_logreg.named_steps["clf"].estimators_
])
test_scores = np.column_stack([
    est.predict_proba(Xt)[:, 1] for est in char_logreg.named_steps["clf"].estimators_
])

char_thresholds = tune_thresholds(y_val, val_scores, LABELS)
char_pred = apply_thresholds(test_scores, char_thresholds, LABELS)

char_summary, char_per_label = evaluate_multilabel(y_test, char_pred, LABELS, "Char n-gram + LogReg")
display(pd.DataFrame([char_summary]))
display(char_per_label)
print("Thresholds:", char_thresholds)

In [ ]:
# =========================
# 12. Classical Model D: Word2Vec average embeddings + Logistic Regression
# =========================
from gensim.models import Word2Vec

def tokenize_for_w2v(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-Z']+", text)

X_train_tok = [tokenize_for_w2v(x) for x in X_train]
X_val_tok   = [tokenize_for_w2v(x) for x in X_val]
X_test_tok  = [tokenize_for_w2v(x) for x in X_test]

w2v = Word2Vec(
    sentences=X_train_tok,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

def avg_w2v(tokens, model, dim=100):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_w2v = np.vstack([avg_w2v(t, w2v, 100) for t in X_train_tok])
X_val_w2v   = np.vstack([avg_w2v(t, w2v, 100) for t in X_val_tok])
X_test_w2v  = np.vstack([avg_w2v(t, w2v, 100) for t in X_test_tok])

w2v_logreg = OneVsRestClassifier(
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear"
    )
)

w2v_logreg.fit(X_train_w2v, y_train)

val_scores = np.column_stack([est.predict_proba(X_val_w2v)[:, 1] for est in w2v_logreg.estimators_])
test_scores = np.column_stack([est.predict_proba(X_test_w2v)[:, 1] for est in w2v_logreg.estimators_])

w2v_thresholds = tune_thresholds(y_val, val_scores, LABELS)
w2v_pred = apply_thresholds(test_scores, w2v_thresholds, LABELS)

w2v_summary, w2v_per_label = evaluate_multilabel(y_test, w2v_pred, LABELS, "Word2Vec + LogReg")
display(pd.DataFrame([w2v_summary]))
display(w2v_per_label)
print("Thresholds:", w2v_thresholds)

In [ ]:
# =========================
# 13. Classical comparison table
# =========================
classical_results = pd.DataFrame([
    count_summary, hash_summary, char_summary, w2v_summary
]).sort_values(["macro_f1", "micro_f1"], ascending=False).reset_index(drop=True)

display(classical_results)

best_classical_name = classical_results.loc[0, "model"]
print("Best classical model:", best_classical_name)

classical_results.to_csv(OUTPUT_DIR / "classical_vectorization_comparison.csv", index=False)

In [ ]:
# =========================
# 14. Plot classical comparison
# =========================
plot_df = classical_results.set_index("model")[["macro_f1", "micro_f1", "subset_accuracy"]]
ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_title("Classical Product Label Prediction Comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "classical_comparison_barplot.png", dpi=200)
plt.show()

## DistilBERT multi-label benchmark

This section adds a transformer model **after** the classical comparison.

The model:
- takes raw text as input
- predicts 5 labels with **sigmoid** outputs
- uses **BCEWithLogitsLoss**
- tunes one threshold per label on the validation set

In [ ]:
# =========================
# 15. Install / import transformer dependencies if needed
# =========================
try:
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception as e:
    print("If this cell fails, install dependencies in terminal or a new cell:")
    print("pip install transformers datasets accelerate")
    raise

In [ ]:
# =========================
# 16. Prepare Hugging Face datasets
# =========================
import torch
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[["text"] + LABELS].reset_index(drop=True))
val_hf   = Dataset.from_pandas(val_df[["text"] + LABELS].reset_index(drop=True))
test_hf  = Dataset.from_pandas(test_df[["text"] + LABELS].reset_index(drop=True))

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_batch(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256
    )
    enc["labels"] = [
        [float(batch[label][i]) for label in LABELS] for i in range(len(batch["text"]))
    ]
    return enc

train_tok = train_hf.map(preprocess_batch, batched=True, remove_columns=train_hf.column_names)
val_tok   = val_hf.map(preprocess_batch, batched=True, remove_columns=val_hf.column_names)
test_tok  = test_hf.map(preprocess_batch, batched=True, remove_columns=test_hf.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# =========================
# 17. Build multi-label DistilBERT model
# =========================
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    problem_type="multi_label_classification"
)

In [ ]:
# =========================
# 18. Metrics helper for Trainer
# =========================
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "micro_f1@0.5": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1@0.5": f1_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
# =========================
# 19. Train DistilBERT
# =========================
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "distilbert_ckpt"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1@0.5",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# =========================
# 20. DistilBERT threshold tuning on validation set
# =========================
val_pred = trainer.predict(val_tok)
test_pred = trainer.predict(test_tok)

val_probs = 1 / (1 + np.exp(-val_pred.predictions))
test_probs = 1 / (1 + np.exp(-test_pred.predictions))

bert_thresholds = tune_thresholds(y_val, val_probs, LABELS)
bert_test_pred = apply_thresholds(test_probs, bert_thresholds, LABELS)

bert_summary, bert_per_label = evaluate_multilabel(
    y_test, bert_test_pred, LABELS, "DistilBERT multi-label"
)

display(pd.DataFrame([bert_summary]))
display(bert_per_label)
print("Thresholds:", bert_thresholds)

In [ ]:
# =========================
# 21. Final comparison: best classical vs DistilBERT
# =========================
final_results = pd.DataFrame([count_summary, hash_summary, char_summary, w2v_summary, bert_summary])
final_results = final_results.sort_values(["macro_f1", "micro_f1"], ascending=False).reset_index(drop=True)

display(final_results)
final_results.to_csv(OUTPUT_DIR / "final_product_label_comparison_with_distilbert.csv", index=False)

best_overall = final_results.loc[0, "model"]
print("Best overall model:", best_overall)

In [ ]:
# =========================
# 22. Plot final comparison
# =========================
plot_df = final_results.set_index("model")[["macro_f1", "micro_f1", "subset_accuracy"]]
ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_title("Final 5-label Product Prediction Comparison")
ax.set_ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_comparison_with_distilbert.png", dpi=200)
plt.show()

In [ ]:
# =========================
# 23. Save prediction examples for error analysis
# =========================
# Pick best classical predictions
pred_map = {
    "Count + LinearSVC": count_pred,
    "Hashing + LinearSVC": hash_pred,
    "Char n-gram + LogReg": char_pred,
    "Word2Vec + LogReg": w2v_pred,
    "DistilBERT multi-label": bert_test_pred
}

best_pred = pred_map[best_overall]

error_rows = []
for i in range(len(test_df)):
    true_labels = [LABELS[j] for j in range(len(LABELS)) if y_test[i, j] == 1]
    pred_labels = [LABELS[j] for j in range(len(LABELS)) if best_pred[i, j] == 1]
    if true_labels != pred_labels:
        error_rows.append({
            "ext_id": test_df.iloc[i]["ext_id"],
            "text": test_df.iloc[i]["text"][:500],
            "true_labels": "|".join(true_labels),
            "pred_labels": "|".join(pred_labels)
        })

error_df = pd.DataFrame(error_rows)
display(error_df.head(20))
error_df.to_csv(OUTPUT_DIR / "best_model_error_examples.csv", index=False)

In [ ]:
# =========================
# 24. Save per-label tables
# =========================
count_per_label.to_csv(OUTPUT_DIR / "count_per_label.csv", index=False)
hash_per_label.to_csv(OUTPUT_DIR / "hash_per_label.csv", index=False)
char_per_label.to_csv(OUTPUT_DIR / "char_per_label.csv", index=False)
w2v_per_label.to_csv(OUTPUT_DIR / "w2v_per_label.csv", index=False)
bert_per_label.to_csv(OUTPUT_DIR / "distilbert_per_label.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)

## Notes

- If DistilBERT is too slow on your machine, lower `num_train_epochs` to 2 and/or reduce `max_length` to 192.
- If you run into memory issues, reduce batch size from 8 to 4.
- The classical and transformer sections are intentionally separated:
  - first compare **classical vectorization methods**
  - then compare the **best overall classical pipeline** against **DistilBERT**